# ASCAT Ultra-Safe Smoke Test

This notebook is a stability check for CDS JupyterLab, not a production backfill.

Design constraints:
- ASCAT only. Recon is intentionally disabled.
- Default to 1 manifest row from 2016.
- Tiny time window and small spatial radius.
- 1 retry only.
- Temp subset files are deleted immediately after inspection.

Acceptable outcomes for this smoke test:
- subset download succeeds and the NetCDF file can be opened;
- the request returns no matching data but the notebook stays alive;
- a controlled Python exception is raised.

If the kernel or server still restarts under this notebook, the ASCAT historical pipeline is very likely not suitable for CDS JupyterLab as the main execution environment.


In [ ]:
import subprocess
import sys


def _pip_install(packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages]
    print('[RUN]', ' '.join(cmd))
    subprocess.check_call(cmd)


_pip_install(['pip'])
_pip_install(['numpy', 'xarray', 'copernicusmarine', 'netCDF4'])
print('dependency bootstrap complete')


In [ ]:
from pathlib import Path

CONFIG = {
    'workdir': '/home/jovyan/mystorage/cds_obs_ultra_safe_smoke',
    'ascat_manifest_input': '/home/jovyan/mystorage/ascat_request_manifest.csv',
    'ascat_username': '',
    'ascat_password': '',
    'ascat_credentials_file': '',
    'ascat_service': '',
    'allow_interactive_credential_prompt': False,
    'year_filter': 2016,
    'smoke_rows': 1,
    'only_with_storm_id': True,
    'request_id_allowlist': [],
    'dataset_ids': [
        'cmems_obs-wind_glo_phy-ascat-metop_b-l3-pt1h'
    ],
    'variables': [],
    'window_before_min': 60,
    'window_after_min': 60,
    'inner_radius_km': 100.0,
    'outer_radius_km': 250.0,
    'bbox_margin_deg': 0.05,
    'max_retries': 1,
    'sleep_sec': 2.0,
    'keep_temp_files': False,
    'tmp_dir': '/tmp/ascat_ultra_safe_smoke',
    'summary_json': 'ascat_ultra_safe_smoke_summary.json',
    'selected_manifest_csv': 'ascat_ultra_safe_selected_manifest.csv',
}

CONFIG_RUNTIME_OVERRIDE = {
    'ascat_username': '',
    'ascat_password': '',
    'request_id_allowlist': [],
    'dataset_ids': [],
}

for key, value in CONFIG_RUNTIME_OVERRIDE.items():
    if value not in ('', None, []):
        CONFIG[key] = value

print({
    'workdir': CONFIG['workdir'],
    'manifest': CONFIG['ascat_manifest_input'],
    'year_filter': CONFIG['year_filter'],
    'smoke_rows': CONFIG['smoke_rows'],
    'dataset_ids': CONFIG['dataset_ids'],
    'window_before_min': CONFIG['window_before_min'],
    'window_after_min': CONFIG['window_after_min'],
    'outer_radius_km': CONFIG['outer_radius_km'],
    'bbox_margin_deg': CONFIG['bbox_margin_deg'],
})


In [ ]:
import csv
import inspect
import json
import math
import os
import time
from datetime import datetime, timedelta, timezone

import copernicusmarine
import numpy as np
import xarray as xr


def parse_iso_utc(value):
    txt = (value or '').strip()
    for fmt in ('%Y-%m-%dT%H:%M:%SZ', '%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S'):
        try:
            return datetime.strptime(txt, fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            continue
    raise ValueError(f'unsupported datetime format: {value}')


def parse_float(value):
    try:
        if value is None:
            return None
        txt = str(value).strip()
        if not txt:
            return None
        return float(txt)
    except Exception:
        return None


def parse_lon_minus180_180(lon):
    out = float(lon)
    while out >= 180:
        out -= 360
    while out < -180:
        out += 360
    return out


def build_bbox(lat, lon, outer_radius_km, margin_deg):
    lat_delta = (outer_radius_km / 111.0) + margin_deg
    cos_lat = max(0.15, abs(math.cos(math.radians(lat))))
    lon_delta = (outer_radius_km / (111.0 * cos_lat)) + margin_deg
    lon_c = parse_lon_minus180_180(lon)
    min_lon = lon_c - lon_delta
    max_lon = lon_c + lon_delta
    min_lat = max(-89.9, lat - lat_delta)
    max_lat = min(89.9, lat + lat_delta)
    return min_lon, max_lon, min_lat, max_lat


def safe_request_id(request_id):
    out = []
    for ch in str(request_id):
        if ch.isalnum() or ch in ('-', '_'):
            out.append(ch)
        else:
            out.append('_')
    return ''.join(out)[:128]


def get_credentials(config):
    username = (
        str(config.get('ascat_username') or '').strip()
        or str(os.getenv('COPERNICUSMARINE_SERVICE_USERNAME') or '').strip()
        or str(os.getenv('COPERNICUSMARINE_USERNAME') or '').strip()
    )
    password = (
        str(config.get('ascat_password') or '').strip()
        or str(os.getenv('COPERNICUSMARINE_SERVICE_PASSWORD') or '').strip()
        or str(os.getenv('COPERNICUSMARINE_PASSWORD') or '').strip()
    )
    credentials_file = str(config.get('ascat_credentials_file') or '').strip()
    service = str(config.get('ascat_service') or '').strip()
    return username, password, credentials_file, service


def write_selected_manifest(rows, out_csv):
    if not rows:
        raise RuntimeError('cannot write empty selected manifest')
    fieldnames = list(rows[0].keys())
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with out_csv.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def load_smoke_rows(config):
    manifest_csv = Path(config['ascat_manifest_input']).resolve()
    if not manifest_csv.exists():
        raise FileNotFoundError(f'manifest not found: {manifest_csv}')

    year_filter = int(config['year_filter'])
    smoke_rows = int(config['smoke_rows'])
    allowlist = set(config.get('request_id_allowlist') or [])
    selected = []

    with manifest_csv.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            issue_time = str(row.get('issue_time_utc') or '').strip()
            if not issue_time.startswith(str(year_filter)):
                continue
            if config.get('only_with_storm_id', True) and not str(row.get('storm_id') or '').strip():
                continue
            request_id = str(row.get('request_id') or '').strip()
            if allowlist and request_id not in allowlist:
                continue
            lat = parse_float(row.get('lat'))
            lon = parse_float(row.get('lon'))
            if lat is None or lon is None:
                continue
            row['_issue_dt'] = parse_iso_utc(issue_time)
            row['_lat_float'] = float(lat)
            row['_lon_float'] = float(lon)
            selected.append(row)
            if len(selected) >= smoke_rows:
                break

    if not selected:
        raise RuntimeError('no qualifying smoke-test rows found in manifest')
    return selected


def call_subset(dataset_id, out_file, min_lon, max_lon, min_lat, max_lat, start_dt, end_dt, config, creds):
    username, password, credentials_file, service = creds
    subset_sig = inspect.signature(copernicusmarine.subset)
    accepted = set(subset_sig.parameters.keys())
    kwargs = {
        'dataset_id': dataset_id,
        'minimum_longitude': float(min_lon),
        'maximum_longitude': float(max_lon),
        'minimum_latitude': float(min_lat),
        'maximum_latitude': float(max_lat),
        'start_datetime': start_dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'end_datetime': end_dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'output_directory': str(out_file.parent),
        'output_filename': out_file.name,
        'overwrite': True,
        'disable_progress_bar': True,
    }
    variables = config.get('variables') or []
    if variables:
        kwargs['variables'] = variables
    if username:
        kwargs['username'] = username
    if password:
        kwargs['password'] = password
    if credentials_file:
        kwargs['credentials_file'] = credentials_file
    if service:
        kwargs['service'] = service

    filtered = {k: v for k, v in kwargs.items() if k in accepted and v not in ('', None, [])}
    result = copernicusmarine.subset(**filtered)

    if out_file.exists() and out_file.stat().st_size > 0:
        return out_file
    if isinstance(result, dict):
        for key in ('output_path', 'file_path', 'path'):
            cand = result.get(key)
            if cand:
                fp = Path(str(cand))
                if fp.exists() and fp.stat().st_size > 0:
                    return fp
        for cand in result.get('files', []):
            fp = Path(str(cand))
            if fp.exists() and fp.stat().st_size > 0:
                return fp

    raise RuntimeError(f'subset call finished but output file is missing: {out_file}')


def pick_wind_variable(ds):
    preferred = ['wind_speed', 'wind_speed_10m', 'windspeed', 'windspeed_10m', 'wind_speed_mean']
    lower_to_name = {name.lower(): name for name in ds.data_vars}
    for item in preferred:
        found = lower_to_name.get(item.lower())
        if found:
            return found
    for name in ds.data_vars:
        lname = name.lower()
        if 'wind' in lname and 'speed' in lname:
            return name
    return next(iter(ds.data_vars), None)


print('helper functions ready')


In [ ]:
WORKDIR = Path(CONFIG['workdir']).resolve()
TMP_DIR = Path(CONFIG['tmp_dir']).resolve()
SUMMARY_PATH = WORKDIR / CONFIG['summary_json']
SELECTED_MANIFEST_PATH = WORKDIR / CONFIG['selected_manifest_csv']
WORKDIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

selected_rows = load_smoke_rows(CONFIG)
write_selected_manifest([{k: v for k, v in row.items() if not k.startswith('_')} for row in selected_rows], SELECTED_MANIFEST_PATH)

creds = get_credentials(CONFIG)
has_noninteractive_credentials = bool((creds[0] and creds[1]) or creds[2])
if not has_noninteractive_credentials and not bool(CONFIG.get('allow_interactive_credential_prompt', False)):
    raise RuntimeError(
        'ASCAT credentials missing. Fill CONFIG ascat_username/ascat_password, or set ascat_credentials_file, '
        'or export COPERNICUSMARINE_SERVICE_USERNAME and COPERNICUSMARINE_SERVICE_PASSWORD.'
    )

print({
    'selected_rows': len(selected_rows),
    'selected_manifest_csv': str(SELECTED_MANIFEST_PATH),
    'summary_json': str(SUMMARY_PATH),
    'username_set': bool(creds[0]),
    'password_set': bool(creds[1]),
    'credentials_file_set': bool(creds[2]),
    'dataset_ids': CONFIG['dataset_ids'],
})

for idx, row in enumerate(selected_rows, start=1):
    print({
        'smoke_row': idx,
        'request_id': row.get('request_id'),
        'issue_time_utc': row.get('issue_time_utc'),
        'storm_id': row.get('storm_id'),
        'lat': row.get('lat'),
        'lon': row.get('lon'),
    })


In [ ]:
results = []

for row_index, row in enumerate(selected_rows, start=1):
    issue_dt = row['_issue_dt']
    lat = row['_lat_float']
    lon = row['_lon_float']
    start_dt = issue_dt - timedelta(minutes=int(CONFIG['window_before_min']))
    end_dt = issue_dt + timedelta(minutes=int(CONFIG['window_after_min']))
    min_lon, max_lon, min_lat, max_lat = build_bbox(
        lat=lat,
        lon=lon,
        outer_radius_km=float(CONFIG['outer_radius_km']),
        margin_deg=float(CONFIG['bbox_margin_deg']),
    )

    print('')
    print(f'========== SMOKE ROW {row_index}/{len(selected_rows)} ==========')
    print({
        'request_id': row.get('request_id'),
        'issue_time_utc': row.get('issue_time_utc'),
        'dataset_ids': CONFIG['dataset_ids'],
        'bbox': {
            'min_lon': round(min_lon, 4),
            'max_lon': round(max_lon, 4),
            'min_lat': round(min_lat, 4),
            'max_lat': round(max_lat, 4),
        },
        'window': {
            'start': start_dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
            'end': end_dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
        },
    })

    row_result = {
        'request_id': row.get('request_id'),
        'issue_time_utc': row.get('issue_time_utc'),
        'status': 'failed',
        'dataset_id': '',
        'subset_file_mb': '',
        'opened_variable': '',
        'error': '',
    }

    for dataset_id in list(CONFIG['dataset_ids']):
        temp_name = f"ascat_smoke_{safe_request_id(row.get('request_id'))}_{dataset_id.replace('/', '_')}.nc"
        temp_path = TMP_DIR / temp_name

        for attempt in range(1, int(CONFIG['max_retries']) + 1):
            print(f'[SUBSET] dataset={dataset_id} attempt={attempt}')
            try:
                subset_path = call_subset(
                    dataset_id=dataset_id,
                    out_file=temp_path,
                    min_lon=min_lon,
                    max_lon=max_lon,
                    min_lat=min_lat,
                    max_lat=max_lat,
                    start_dt=start_dt,
                    end_dt=end_dt,
                    config=CONFIG,
                    creds=creds,
                )

                subset_size_mb = round(subset_path.stat().st_size / 1024.0 / 1024.0, 3)
                print({'subset_path': str(subset_path), 'subset_file_mb': subset_size_mb})

                ds = xr.open_dataset(subset_path, cache=False)
                try:
                    variable_name = pick_wind_variable(ds)
                    if variable_name is None:
                        raise RuntimeError('dataset opened but no data variables were found')
                    da = ds[variable_name]
                    print({
                        'opened_variable': variable_name,
                        'units': str(da.attrs.get('units') or ''),
                        'data_vars': list(ds.data_vars),
                        'sizes': {k: int(v) for k, v in da.sizes.items()},
                        'coords': list(da.coords),
                    })
                finally:
                    ds.close()

                row_result = {
                    'request_id': row.get('request_id'),
                    'issue_time_utc': row.get('issue_time_utc'),
                    'status': 'opened_dataset',
                    'dataset_id': dataset_id,
                    'subset_file_mb': subset_size_mb,
                    'opened_variable': variable_name,
                    'error': '',
                }
                break
            except Exception as exc:
                last_error = f'{type(exc).__name__}: {str(exc)[:240]}'
                print('[WARN]', last_error)
                row_result['error'] = last_error
                if attempt < int(CONFIG['max_retries']):
                    time.sleep(min(5.0, float(CONFIG['sleep_sec'])))
            finally:
                if not bool(CONFIG.get('keep_temp_files', False)) and temp_path.exists():
                    try:
                        temp_path.unlink()
                    except Exception:
                        pass

        if row_result['status'] == 'opened_dataset':
            break

    results.append(row_result)
    time.sleep(max(0.0, float(CONFIG['sleep_sec'])))

summary = {
    'generated_at_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'workdir': str(WORKDIR),
    'selected_manifest_csv': str(SELECTED_MANIFEST_PATH),
    'rows_attempted': len(results),
    'rows_opened_dataset': sum(1 for item in results if item['status'] == 'opened_dataset'),
    'rows_failed': sum(1 for item in results if item['status'] != 'opened_dataset'),
    'config_snapshot': {
        'year_filter': CONFIG['year_filter'],
        'smoke_rows': CONFIG['smoke_rows'],
        'dataset_ids': CONFIG['dataset_ids'],
        'window_before_min': CONFIG['window_before_min'],
        'window_after_min': CONFIG['window_after_min'],
        'inner_radius_km': CONFIG['inner_radius_km'],
        'outer_radius_km': CONFIG['outer_radius_km'],
        'bbox_margin_deg': CONFIG['bbox_margin_deg'],
        'max_retries': CONFIG['max_retries'],
        'sleep_sec': CONFIG['sleep_sec'],
    },
    'results': results,
}

SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('')
print(SUMMARY_PATH)
print(json.dumps(summary, ensure_ascii=False, indent=2))

if summary['rows_opened_dataset'] > 0:
    print('Smoke test reached Copernicus Marine subset + xarray open_dataset successfully.')
else:
    print('No dataset was opened. If the kernel stayed alive, this is still a useful stability signal; inspect the per-row errors above.')


## If This Notebook Stays Stable

Increase load one axis at a time:
1. keep `smoke_rows=1`, but switch `dataset_ids` to two or three satellites;
2. then move `smoke_rows` from `1` to `3`;
3. then widen the time window to `120` minutes;
4. only after that try the larger chunked notebook again.
